In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from boruta import BorutaPy
from sklearn.preprocessing import LabelEncoder
import warnings

# Suppress warnings for cleaner console output
warnings.filterwarnings('ignore')

print("🚀 Starting the Biological Feature Selection Pipeline...")

# ==========================================
# PHASE 0: DATA LOADING & PREPARATION 
# ==========================================
print("\n[Phase 0] Loading Data & Preserving Sample Codes...")
df_merged = pd.read_csv('/Users/pawanpahune/AI_Pipeline_Metagenomics/merging_pipeline/Merged_Taxa_Metadata.csv')

# --- THE FIX: Extract and safely store the Sample IDs before dropping ---
sample_codes = df_merged['Code'] 

# Separate Features and Target for math operations
X_raw = df_merged.drop(columns=['Code', 'Target_Crop_Grouped'])
y_raw = df_merged['Target_Crop_Grouped']

# Encode target labels to strict 0, 1, 2... format for XGBoost
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

# Apply Half-Min Rule and CLR Transformation
min_non_zero = X_raw[X_raw > 0].min().min()
pseudo_count = min_non_zero / 2
X_pseudo = X_raw + pseudo_count
geom_mean = np.exp(np.mean(np.log(X_pseudo), axis=1))
X_clr = np.log(X_pseudo.div(geom_mean, axis=0))

print(f"Initial Feature Count: {X_clr.shape[1]} Microbes")

# ==========================================
# PHASE 1: THE COLLINEARITY PURGE
# ==========================================
print("\n[Phase 1] Purging Redundant/Collinear Microbes...")
corr_matrix = X_clr.corr(method='spearman').abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]
X_phase1 = X_clr.drop(columns=to_drop)

print(f"Dropped {len(to_drop)} highly correlated microbes.")
print(f"Features remaining: {X_phase1.shape[1]}")

# ==========================================
# PHASE 2: BORUTA (WITH DYNAMIC RESCUE POOL)
# ==========================================
print("\n[Phase 2] Running Boruta Algorithm to eliminate statistical noise...")

xgb_boruta = xgb.XGBClassifier(
    max_depth=3, learning_rate=0.05, n_estimators=100, random_state=42, n_jobs=-1
)

boruta_selector = BorutaPy(
    xgb_boruta, n_estimators='auto', verbose=0, random_state=42, max_iter=50 
)
boruta_selector.fit(X_phase1.values, y)

boruta_strong = X_phase1.columns[boruta_selector.support_].tolist()
boruta_tentative = X_phase1.columns[boruta_selector.support_weak_].tolist()
combined_boruta = list(set(boruta_strong + boruta_tentative))

target_feature_count = 15

if len(combined_boruta) <= target_feature_count:
    print(f"⚠️ Boruta only validated {len(combined_boruta)} microbes. Activating Dynamic Rescue Pool...")
    xgb_baseline = xgb.XGBClassifier(max_depth=3, random_state=42, n_jobs=-1)
    xgb_baseline.fit(X_phase1, y)
    importances = pd.Series(xgb_baseline.feature_importances_, index=X_phase1.columns)
    top_baseline_features = importances.sort_values(ascending=False).head(40).index.tolist()
    pool_features = list(set(combined_boruta + top_baseline_features))
    X_phase2 = X_phase1[pool_features]
    print(f"Rescue Pool created with {X_phase2.shape[1]} candidate microbes.")
else:
    X_phase2 = X_phase1[combined_boruta]
    print(f"Features remaining after Boruta: {X_phase2.shape[1]}")

# ==========================================
# PHASE 3: SHAP-GUIDED RECURSIVE ELIMINATION
# ==========================================
print("\n[Phase 3] Running SHAP-RFE to isolate the top 15 Elite Microbes...")

current_features = list(X_phase2.columns)
xgb_shap = xgb.XGBClassifier(
    max_depth=3, learning_rate=0.05, n_estimators=100, random_state=42, n_jobs=-1
)

if len(current_features) > target_feature_count:
    while len(current_features) > target_feature_count:
        X_subset = X_phase2[current_features]
        xgb_shap.fit(X_subset, y)
        explainer = shap.TreeExplainer(xgb_shap)
        shap_values = explainer.shap_values(X_subset)
        
        if isinstance(shap_values, list):
            shap_sum = np.zeros(X_subset.shape[1])
            for sv in shap_values:
                shap_sum += np.abs(sv).mean(axis=0)
        elif len(shap_values.shape) == 3:
            shap_sum = np.abs(shap_values).mean(axis=(0, 2))
        else:
            shap_sum = np.abs(shap_values).mean(axis=0)
            
        least_important_idx = np.argmin(shap_sum)
        current_features.remove(current_features[least_important_idx])

# ==========================================
# PHASE 4: EXPORT WITH SAMPLE CODES (FIXED)
# ==========================================
print("\n[Phase 4] Creating Final Dataset with Metadata Triggers...")

# Create the final dataframe with the 15 elite features
final_df = X_clr[current_features].copy()

# --- THE FIX: Snap the 'Code' column back to the front of the dataset ---
final_df.insert(0, 'Code', sample_codes.values)

# Add the Target column back at the end
final_df['Target_Crop_Grouped'] = y_raw.values 

# Save to CSV
output_filename = 'compressed_df_final.csv'
final_df.to_csv(output_filename, index=False)

print(f"✅ SUCCESS: File saved locally as '{output_filename}'")
print(f"Final Data Shape: {final_df.shape[0]} rows, {final_df.shape[1]} columns")
print(f"Columns included: [Code] + [15 Microbes] + [Target_Crop_Grouped]")

🚀 Starting the Biological Feature Selection Pipeline...

[Phase 0] Loading Data & Preserving Sample Codes...
Initial Feature Count: 42 Microbes

[Phase 1] Purging Redundant/Collinear Microbes...
Dropped 4 highly correlated microbes.
Features remaining: 38

[Phase 2] Running Boruta Algorithm to eliminate statistical noise...
⚠️ Boruta only validated 7 microbes. Activating Dynamic Rescue Pool...
Rescue Pool created with 38 candidate microbes.

[Phase 3] Running SHAP-RFE to isolate the top 15 Elite Microbes...

[Phase 4] Creating Final Dataset with Metadata Triggers...
✅ SUCCESS: File saved locally as 'compressed_df_final.csv'
Final Data Shape: 67 rows, 17 columns
Columns included: [Code] + [15 Microbes] + [Target_Crop_Grouped]


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import warnings

# Suppress warnings for a clean console output
warnings.filterwarnings('ignore')

print("🚀 Initiating Metadata Feature Extraction...")

# ==========================================
# PHASE 1: LOAD DATA & LOCK THE INDEX
# ==========================================
# Locking 'Code' to the index guarantees rows cannot be shuffled or misaligned
df_meta = pd.read_csv('/Users/pawanpahune/AI_Pipeline_Metagenomics/Metadata-without-taxa-v1.csv').set_index('Code')

# Separate Features and Target
X = df_meta.drop(columns=['Target_Crop_Grouped'])

# Extract target as strictly integers (no LabelEncoder needed)
y = df_meta['Target_Crop_Grouped'].astype(int).values.ravel()

# ==========================================
# PHASE 2: SHAP-RFE FEATURE SELECTION
# ==========================================
current_features = list(X.columns)
target_count = 10

# Initialize a shallow tree model. 
# Shallow depth prevents the model from overfitting to noise during the selection process.
xgb_shap = xgb.XGBClassifier(
    max_depth=3, 
    learning_rate=0.05, 
    n_estimators=100, 
    random_state=42, 
    n_jobs=-1
)

print(f"Starting with {len(current_features)} features. Isolating the top {target_count}...")

# Recursively eliminate the weakest feature based on SHAP values
while len(current_features) > target_count:
    X_subset = X[current_features]
    xgb_shap.fit(X_subset, y)
    
    # Calculate Shapley values for the current subset
    explainer = shap.TreeExplainer(xgb_shap)
    shap_values = explainer.shap_values(X_subset)
    
    # Safely handle multi-class SHAP array dimensions
    if isinstance(shap_values, list):
        shap_sum = np.zeros(X_subset.shape[1])
        for sv in shap_values:
            shap_sum += np.abs(sv).mean(axis=0)
    elif len(shap_values.shape) == 3:
        shap_sum = np.abs(shap_values).mean(axis=(0, 2))
    else:
        shap_sum = np.abs(shap_values).mean(axis=0)
        
    # Identify and drop the feature with the lowest overall impact
    least_important_idx = np.argmin(shap_sum)
    current_features.remove(current_features[least_important_idx])

print("\n✅ Top 10 Agricultural Drivers Identified:")
for i, feature in enumerate(current_features, 1):
    print(f"  {i}. {feature}")

# ==========================================
# PHASE 3: CONSTRUCT FINAL DATASET
# ==========================================
print("\n💾 Constructing 'df_final_metadata'...")

# Isolate the winning features
df_final_metadata = X[current_features].copy()

# Append the Target variable
df_final_metadata['Target_Crop_Grouped'] = y

# Release the 'Code' index back into a standard column (Column 0)
df_final_metadata.reset_index(inplace=True)

# Export to CSV
output_filename = 'df_final_metadata.csv'
df_final_metadata.to_csv(output_filename, index=False)

print(f"✅ Success: File saved locally as '{output_filename}'")
print(f"Shape: {df_final_metadata.shape[0]} rows, {df_final_metadata.shape[1]} columns")
print("Structure: [Code] + [10 Elite Features] + [Target_Crop_Grouped]")

🚀 Initiating Metadata Feature Extraction...
Starting with 23 features. Isolating the top 10...

✅ Top 10 Agricultural Drivers Identified:
  1. Irrigation_Usage
  2. Organic_Practices
  3. Biodynamic_Methods
  4. Total_P_Applied
  5. Total_K_Applied
  6. Total_Zinc_Applied
  7. Total_Sulfur_Applied
  8. Previous_Year_Crop_Frijol
  9. Irrigation_Type_Superficie
  10. Product_1_Type_Fertilizante sintético

💾 Constructing 'df_final_metadata'...
✅ Success: File saved locally as 'df_final_metadata.csv'
Shape: 67 rows, 12 columns
Structure: [Code] + [10 Elite Features] + [Target_Crop_Grouped]


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, balanced_accuracy_score
from lazypredict.Supervised import LazyClassifier
import warnings

warnings.filterwarnings('ignore')

print("📂 [Step 1] Loading and Merging Datasets...")

# Load the files
df_microbes = pd.read_csv('/Users/pawanpahune/AI_Pipeline_Metagenomics/df_compression/compressed_df_final.csv')
df_meta = pd.read_csv('/Users/pawanpahune/AI_Pipeline_Metagenomics/df_compression/df_final_metadata.csv')

# Drop the target from the metadata so we don't have duplicates when merging
df_meta_clean = df_meta.drop(columns=['Target_Crop_Grouped'])

# Merge strictly on the 'Code' column to preserve ID alignment
df_merged = pd.merge(df_microbes, df_meta_clean, on='Code', how='inner')

# Save the master dataset for future ML tracking
output_filename = 'Master_Combined_Dataset.csv'
df_merged.to_csv(output_filename, index=False)

print(f"✅ Merged Successfully! Final shape: {df_merged.shape}")
print(f"✅ Master dataset saved as: {output_filename}")

# ==========================================
# [Step 2] Data Prep for LazyPredict
# ==========================================
X = df_merged.drop(columns=['Code', 'Target_Crop_Grouped'])
y = df_merged['Target_Crop_Grouped'].astype(int)

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ==========================================
# [Step 3] Custom LazyPredict Execution
# ==========================================
print("\n🚀 [Step 3] Running LazyPredict with all Custom Metrics...")

# Initialize LazyClassifier
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)

# Fit the models
models, predictions = clf.fit(X_train, X_test, y_train, y_test)

# --- Adding Precision & Recall Manually ---
# LazyPredict returns 'Accuracy', 'Balanced Accuracy', 'ROC AUC', 'F1 Score' natively.
# We map through its dictionary of trained models to extract Precision and Recall
precision_list = []
recall_list = []

for model_name in models.index:
    try:
        # Get the actual trained sklearn model from LazyPredict's backend
        trained_model = clf.models[model_name]
        
        # Make predictions on the test set
        y_pred = trained_model.predict(X_test)
        
        # Calculate weighted precision and recall
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted')
        
        precision_list.append(prec)
        recall_list.append(rec)
    except Exception as e:
        precision_list.append(None)
        recall_list.append(None)

# Attach the new metrics to the LazyPredict results table
models['Precision'] = precision_list
models['Recall'] = recall_list

# Save to CSV
models.to_csv('LazyPredict_Combined_Results.csv')

print("\n🏆 Final LazyPredict Evaluation:")
# Rearranging columns for cleaner viewing
display_cols = ['Accuracy', 'Balanced Accuracy', 'F1 Score', 'Precision', 'Recall']
print(models[display_cols]))

2026/06/06 05:23:23 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2026/06/06 05:23:23 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


📂 [Step 1] Loading and Merging Datasets...
✅ Merged Successfully! Final shape: (67, 27)
✅ Master dataset saved as: Master_Combined_Dataset.csv

🚀 [Step 3] Running LazyPredict with all Custom Metrics...


2026/06/06 05:23:27 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2026/06/06 05:23:27 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.



🏆 Final LazyPredict Evaluation:


KeyError: "['Accuracy', 'Balanced Accuracy', 'F1 Score'] not in index"

In [6]:
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, balanced_accuracy_score

# Import all top tier classifiers
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb

warnings.filterwarnings('ignore')

print("📂 [Step 1] Loading and Merging Datasets...")

# Load the files
df_microbes = pd.read_csv('/Users/pawanpahune/AI_Pipeline_Metagenomics/df_compression/compressed_df_final.csv')
df_meta = pd.read_csv('/Users/pawanpahune/AI_Pipeline_Metagenomics/df_compression/df_final_metadata.csv')

# Drop the target from the metadata so we don't have duplicates when merging
df_meta_clean = df_meta.drop(columns=['Target_Crop_Grouped'])

# Merge strictly on the 'Code' column to preserve ID alignment
df_merged = pd.merge(df_microbes, df_meta_clean, on='Code', how='inner')

# Prepare X and y
X = df_merged.drop(columns=['Code', 'Target_Crop_Grouped'])
y = df_merged['Target_Crop_Grouped'].astype(int)

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.50, random_state=42, stratify=y
)

print(f"✅ Data Ready! Training shape: {X_train.shape}, Test shape: {X_test.shape}")
print("\n🚀 [Step 2] Running Custom Model Evaluator (Safe Multi-Class)...\n")

# Dictionary of all the models we want to test
models = {
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(random_state=42, class_weight='balanced'),
    'Ridge Classifier': RidgeClassifier(random_state=42, class_weight='balanced'),
    'Bagging': BaggingClassifier(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'SVC': SVC(random_state=42, class_weight='balanced'),
    'Linear SVC': LinearSVC(random_state=42, class_weight='balanced'),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000),
    'K-Neighbors': KNeighborsClassifier(),
    'Gaussian NB': GaussianNB()
}

results = []

# Loop through each model, train, predict, and score
for name, model in models.items():
    try:
        # Train the model
        model.fit(X_train, y_train)
        
        # Predict on holdout test set
        y_pred = model.predict(X_test)
        
        # Calculate strict multi-class metrics
        acc = accuracy_score(y_test, y_pred)
        b_acc = balanced_accuracy_score(y_test, y_pred)
        
        # We use average='weighted' to account for class imbalances in multi-class data
        f1 = f1_score(y_test, y_pred, average='weighted')
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted')
        
        results.append({
            'Model': name,
            'Accuracy': acc,
            'Balanced Accuracy': b_acc,
            'F1 Score': f1,
            'Precision': prec,
            'Recall': rec
        })
    except Exception as e:
        print(f"⚠️ Model '{name}' failed: {e}")

# Compile results into a clean DataFrame
df_results = pd.DataFrame(results)

# Sort by F1 Score (The most robust metric for multi-class)
df_results = df_results.sort_values(by='F1 Score', ascending=False).reset_index(drop=True)

# Save the safe results to CSV
output_filename = 'Custom_Model_Comparison.csv'
df_results.to_csv(output_filename, index=False)

print("🏆 Final Evaluation Results:")
# Format the display to 4 decimal places for readability
print(df_results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

📂 [Step 1] Loading and Merging Datasets...
✅ Data Ready! Training shape: (33, 25), Test shape: (34, 25)

🚀 [Step 2] Running Custom Model Evaluator (Safe Multi-Class)...



2026/06/06 05:30:27 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''
2026/06/06 05:30:27 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during xgboost autologging: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''
2026/06/06 05:30:28 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''
2026/06/06 05:30:28 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''
2026/06/06 05:30:29 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologg

🏆 Final Evaluation Results:
              Model  Accuracy  Balanced Accuracy  F1 Score  Precision  Recall
   Ridge Classifier    0.9412             0.9365    0.9416     0.9502  0.9412
      Random Forest    0.8824             0.8651    0.8785     0.9137  0.8824
  Gradient Boosting    0.8824             0.8651    0.8785     0.9137  0.8824
            Bagging    0.8824             0.8651    0.8785     0.9137  0.8824
                SVC    0.8529             0.8424    0.8550     0.8739  0.8529
Logistic Regression    0.8529             0.8424    0.8529     0.8652  0.8529
         Linear SVC    0.8529             0.8582    0.8520     0.8739  0.8529
        Extra Trees    0.8529             0.8424    0.8498     0.8818  0.8529
            XGBoost    0.8529             0.8294    0.8422     0.8989  0.8529
      Decision Tree    0.8235             0.8196    0.8212     0.8475  0.8235
        Gaussian NB    0.7059             0.7468    0.6748     0.7042  0.7059
        K-Neighbors    0.6471       

In [8]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
import warnings

warnings.filterwarnings('ignore')

print("🕵️‍♂️ INITIATING DATA INTEGRITY & LEAKAGE AUDIT...\n")

# Load your combined dataset
try:
    df = pd.read_csv('/Users/pawanpahune/AI_Pipeline_Metagenomics/df_compression/Master_Combined_Dataset.csv')
except FileNotFoundError:
    print("Error: Could not find 'Master_Combined_Dataset.csv'.")
    exit()

X = df.drop(columns=['Code', 'Target_Crop_Grouped'])
y = df['Target_Crop_Grouped'].astype(int)

# ==========================================
# TEST 1: THE "PERFECT PROXY" CORRELATION TEST
# ==========================================
print("--- TEST 1: Feature-Target Correlation ---")
print("If any feature has a correlation > 0.85 with the target, it is likely a leaked proxy.\n")

# Calculate Spearman correlation with the target
correlations = X.apply(lambda col: col.corr(y, method='spearman')).abs()
leaky_features = correlations[correlations > 0.70].sort_values(ascending=False)

if len(leaky_features) > 0:
    print("🚨 HIGH LEAKAGE RISK DETECTED:")
    for feat, corr in leaky_features.items():
        print(f"   - {feat}: {corr:.4f} correlation")
else:
    print("✅ No massive linear correlations detected.")

# ==========================================
# TEST 2: THE "SINGLE SPLIT" DECISION TREE
# ==========================================
print("\n--- TEST 2: Single-Feature Predictive Power ---")
print("If a Decision Tree with a depth of ONE (a single if/else rule) can get > 80% accuracy, your data is too easy/leaked.\n")

# A 'Decision Stump' can only look at ONE feature to make a guess
stump = DecisionTreeClassifier(max_depth=1, random_state=42)

for col in X.columns:
    X_single = X[[col]]
    
    # 5-Fold Cross Validation on just a single column
    scores = cross_val_score(stump, X_single, y, cv=5, scoring='accuracy')
    mean_acc = scores.mean()
    
    if mean_acc > 0.75:
        print(f"🚨 LEAK WARNING: The column '{col}' ALONE gives {mean_acc*100:.1f}% accuracy.")

# ==========================================
# TEST 3: CATEGORICAL OVERLAP (CROSSTAB)
# ==========================================
print("\n--- TEST 3: Categorical Tautologies ---")
print("Checking if a specific metadata flag uniquely identifies a single crop...\n")

# Look at metadata columns (assuming they have fewer unique values, like integers)
meta_cols = [col for col in X.columns if X[col].nunique() < 10]

for col in meta_cols:
    crosstab = pd.crosstab(X[col], y)
    
    # Check if any column value perfectly points to only one target class
    for index, row in crosstab.iterrows():
        total_in_row = row.sum()
        max_in_class = row.max()
        
        # If a certain farming practice happens >5 times, and 100% of the time it's the exact same crop
        if total_in_row > 5 and (max_in_class / total_in_row) == 1.0:
            dominant_crop = row.idxmax()
            print(f"🚨 TAUTOLOGY: When '{col}' is {index}, the crop is ALWAYS Crop {dominant_crop} ({total_in_row}/{total_in_row} times).")

print("\n🏁 AUDIT COMPLETE.")
print("If you see multiple 🚨 warnings above, your model is not 'smart'; it is just reading the leaked proxies.")

🕵️‍♂️ INITIATING DATA INTEGRITY & LEAKAGE AUDIT...

--- TEST 1: Feature-Target Correlation ---
If any feature has a correlation > 0.85 with the target, it is likely a leaked proxy.

🚨 HIGH LEAKAGE RISK DETECTED:
   - Previous_Year_Crop_Frijol: 0.7189 correlation

--- TEST 2: Single-Feature Predictive Power ---
If a Decision Tree with a depth of ONE (a single if/else rule) can get > 80% accuracy, your data is too easy/leaked.


--- TEST 3: Categorical Tautologies ---
Checking if a specific metadata flag uniquely identifies a single crop...

🚨 TAUTOLOGY: When 'Total_Zinc_Applied' is 2, the crop is ALWAYS Crop 1 (9/9 times).
🚨 TAUTOLOGY: When 'Total_Zinc_Applied' is 3, the crop is ALWAYS Crop 3 (8/8 times).
🚨 TAUTOLOGY: When 'Total_Sulfur_Applied' is 8, the crop is ALWAYS Crop 1 (8/8 times).
🚨 TAUTOLOGY: When 'Previous_Year_Crop_Frijol' is 1, the crop is ALWAYS Crop 3 (15/15 times).

🏁 AUDIT COMPLETE.
If you see multiple 🚨 warnings above, your model is not 'smart'; it is just reading the 

In [9]:
import pandas as pd
import warnings
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, balanced_accuracy_score

# Import all top-tier classifiers
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb

warnings.filterwarnings('ignore')

print("🧹 [Step 1] Loading Data & Surgically Removing Leaks...")

# Load the master dataset
try:
    df = pd.read_csv('/Users/pawanpahune/AI_Pipeline_Metagenomics/df_compression/Master_Combined_Dataset.csv')
except FileNotFoundError:
    print("Error: Could not find 'Master_Combined_Dataset.csv'. Please ensure it's in the same folder.")
    exit()

# ---------------------------------------------------------
# THE FIX: Drop the leaky columns identified by the Audit
# ---------------------------------------------------------
leaky_columns = [
    'Previous_Year_Crop_Frijol', 
    'Total_Zinc_Applied', 
    'Total_Sulfur_Applied'
]

# Drop them, ignoring errors if they are already dropped
df_sanitized = df.drop(columns=leaky_columns, errors='ignore')

# Save the clean dataset so we can use it for hyperparameter tuning later
df_sanitized.to_csv('Sanitized_Master_Dataset.csv', index=False)
print(f"✅ Removed {len(leaky_columns)} 'cheat' columns. Saved as 'Sanitized_Master_Dataset.csv'.")

# ==========================================
# 2. PREPARE X & Y
# ==========================================
X = df_sanitized.drop(columns=['Code', 'Target_Crop_Grouped'])
y = df_sanitized['Target_Crop_Grouped'].astype(int)

# 70/30 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"📊 Training Data: {X_train.shape[0]} farms | Testing Data: {X_test.shape[0]} farms")
print(f"🧬 Honest Features Remaining: {X_train.shape[1]}\n")

# ==========================================
# 3. RUN THE BULLETPROOF "LAZYPREDICT" EVALUATOR
# ==========================================
print("🚀 [Step 2] Running Honest Baseline Evaluation Loop...\n")

models = {
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss'),
    'Extra Trees': ExtraTreesClassifier(random_state=42, class_weight='balanced'),
    'Ridge Classifier': RidgeClassifier(random_state=42, class_weight='balanced'),
    'Bagging': BaggingClassifier(random_state=42),
    'SVC': SVC(random_state=42, class_weight='balanced'),
    'Linear SVC': LinearSVC(random_state=42, class_weight='balanced', max_iter=2000),
    'Logistic Regression': LogisticRegression(random_state=42, class_weight='balanced', max_iter=2000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'K-Neighbors': KNeighborsClassifier(),
    'Gaussian NB': GaussianNB(),
    'AdaBoost': AdaBoostClassifier(random_state=42)
}

results = []

for name, model in models.items():
    try:
        start_time = time.time()
        
        # Train on the sanitized data
        model.fit(X_train, y_train)
        
        # Predict on the holdout test set
        y_pred = model.predict(X_test)
        
        time_taken = time.time() - start_time
        
        # Calculate strict multi-class metrics safely
        acc = accuracy_score(y_test, y_pred)
        b_acc = balanced_accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted')
        
        results.append({
            'Model': name,
            'Accuracy': acc,
            'Balanced Accuracy': b_acc,
            'F1 Score': f1,
            'Precision': prec,
            'Recall': rec,
            'Time (s)': time_taken
        })
    except Exception as e:
        print(f"⚠️ Model '{name}' failed: {e}")

# ==========================================
# 4. VIEW THE HONEST LEADERBOARD
# ==========================================
# Compile and sort by F1 Score (Best metric for imbalanced multi-class)
df_results = pd.DataFrame(results).sort_values(by='F1 Score', ascending=False).reset_index(drop=True)

# Save to CSV
df_results.to_csv('Sanitized_Pipeline_Leaderboard.csv', index=False)

print("🏆 HONEST PIPELINE LEADERBOARD (No Leaks)")
print(df_results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

🧹 [Step 1] Loading Data & Surgically Removing Leaks...
✅ Removed 3 'cheat' columns. Saved as 'Sanitized_Master_Dataset.csv'.
📊 Training Data: 46 farms | Testing Data: 21 farms
🧬 Honest Features Remaining: 22

🚀 [Step 2] Running Honest Baseline Evaluation Loop...



2026/06/06 05:58:09 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''
2026/06/06 05:58:09 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''
2026/06/06 05:58:10 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during xgboost autologging: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''
2026/06/06 05:58:10 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''
2026/06/06 05:58:10 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologg

🏆 HONEST PIPELINE LEADERBOARD (No Leaks)
              Model  Accuracy  Balanced Accuracy  F1 Score  Precision  Recall  Time (s)
   Ridge Classifier    0.9524             0.9643    0.9527     0.9603  0.9524    0.3925
        Extra Trees    0.9524             0.9500    0.9519     0.9603  0.9524    0.3433
           AdaBoost    0.8571             0.8643    0.8599     0.9107  0.8571    0.3412
  Gradient Boosting    0.8571             0.8375    0.8514     0.8903  0.8571    0.4629
            XGBoost    0.8095             0.8286    0.8175     0.8503  0.8095    0.5524
                SVC    0.8095             0.8018    0.8147     0.8373  0.8095    0.4870
      Random Forest    0.8095             0.8161    0.8145     0.8552  0.8095    1.0377
            Bagging    0.8095             0.8018    0.8109     0.8214  0.8095    0.3910
         Linear SVC    0.8095             0.8018    0.8109     0.8214  0.8095    0.4070
      Decision Tree    0.7619             0.7518    0.7664     0.7908  0.7619  

In [10]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
import warnings

warnings.filterwarnings('ignore')

print("🕵️‍♂️ INITIATING DATA INTEGRITY & LEAKAGE AUDIT...\n")

# Load your combined dataset
try:
    df = pd.read_csv('/Users/pawanpahune/AI_Pipeline_Metagenomics/df_compression/Sanitized_Master_Dataset.csv')
except FileNotFoundError:
    print("Error: Could not find 'Master_Combined_Dataset.csv'.")
    exit()

X = df.drop(columns=['Code', 'Target_Crop_Grouped'])
y = df['Target_Crop_Grouped'].astype(int)

# ==========================================
# TEST 1: THE "PERFECT PROXY" CORRELATION TEST
# ==========================================
print("--- TEST 1: Feature-Target Correlation ---")
print("If any feature has a correlation > 0.85 with the target, it is likely a leaked proxy.\n")

# Calculate Spearman correlation with the target
correlations = X.apply(lambda col: col.corr(y, method='spearman')).abs()
leaky_features = correlations[correlations > 0.70].sort_values(ascending=False)

if len(leaky_features) > 0:
    print("🚨 HIGH LEAKAGE RISK DETECTED:")
    for feat, corr in leaky_features.items():
        print(f"   - {feat}: {corr:.4f} correlation")
else:
    print("✅ No massive linear correlations detected.")

# ==========================================
# TEST 2: THE "SINGLE SPLIT" DECISION TREE
# ==========================================
print("\n--- TEST 2: Single-Feature Predictive Power ---")
print("If a Decision Tree with a depth of ONE (a single if/else rule) can get > 80% accuracy, your data is too easy/leaked.\n")

# A 'Decision Stump' can only look at ONE feature to make a guess
stump = DecisionTreeClassifier(max_depth=1, random_state=42)

for col in X.columns:
    X_single = X[[col]]
    
    # 5-Fold Cross Validation on just a single column
    scores = cross_val_score(stump, X_single, y, cv=5, scoring='accuracy')
    mean_acc = scores.mean()
    
    if mean_acc > 0.75:
        print(f"🚨 LEAK WARNING: The column '{col}' ALONE gives {mean_acc*100:.1f}% accuracy.")

# ==========================================
# TEST 3: CATEGORICAL OVERLAP (CROSSTAB)
# ==========================================
print("\n--- TEST 3: Categorical Tautologies ---")
print("Checking if a specific metadata flag uniquely identifies a single crop...\n")

# Look at metadata columns (assuming they have fewer unique values, like integers)
meta_cols = [col for col in X.columns if X[col].nunique() < 10]

for col in meta_cols:
    crosstab = pd.crosstab(X[col], y)
    
    # Check if any column value perfectly points to only one target class
    for index, row in crosstab.iterrows():
        total_in_row = row.sum()
        max_in_class = row.max()
        
        # If a certain farming practice happens >5 times, and 100% of the time it's the exact same crop
        if total_in_row > 5 and (max_in_class / total_in_row) == 1.0:
            dominant_crop = row.idxmax()
            print(f"🚨 TAUTOLOGY: When '{col}' is {index}, the crop is ALWAYS Crop {dominant_crop} ({total_in_row}/{total_in_row} times).")

print("\n🏁 AUDIT COMPLETE.")
print("If you see multiple 🚨 warnings above, your model is not 'smart'; it is just reading the leaked proxies.")

🕵️‍♂️ INITIATING DATA INTEGRITY & LEAKAGE AUDIT...

--- TEST 1: Feature-Target Correlation ---
If any feature has a correlation > 0.85 with the target, it is likely a leaked proxy.

✅ No massive linear correlations detected.

--- TEST 2: Single-Feature Predictive Power ---
If a Decision Tree with a depth of ONE (a single if/else rule) can get > 80% accuracy, your data is too easy/leaked.


--- TEST 3: Categorical Tautologies ---
Checking if a specific metadata flag uniquely identifies a single crop...


🏁 AUDIT COMPLETE.
If you see multiple 🚨 warnings above, your model is not 'smart'; it is just reading the leaked proxies.
